# Q1(c) — implementing the marginal likelihood  ·  5 marks

*Scratchpad, not submission material.*

> Assume $\lambda=1, a=2, b=2$. Write Python code to: (i) compute the log marginal likelihood
> under each of $M_1,M_2,M_3$; (ii) compute the posterior probability of each model under
> equal prior model probabilities; (iii) identify the preferred model.

This part is marked on **working code**, so this notebook deliberately does not implement the
formula — that is the thing being assessed. It covers the two things that actually cost people
marks here: numerical traps, and how to tell whether your implementation is right.

Get the marginal likelihood formula from the notes (the question says it is given there).

## Trap 1 — everything must stay in logs

The question asks for the **log** marginal likelihood, and there is a reason.

In [ ]:
import numpy as np
from scipy.special import gammaln

# Gamma functions overflow long before their logs do.
for a_n in (7, 100, 200):
    try:
        from math import gamma
        direct = gamma(a_n)
    except OverflowError:
        direct = float("inf")
    print(f"a_n = {a_n:4d}   gamma(a_n) = {direct:12.4g}   gammaln(a_n) = {gammaln(a_n):.4f}")

In [ ]:
# Determinants do the same. Use slogdet, which returns (sign, log|det|).
rng = np.random.default_rng(0)
for p in (2, 50, 200):
    A = rng.standard_normal((p, p))
    V = A @ A.T + p * np.eye(p)
    sign, logdet = np.linalg.slogdet(V)
    print(f"p = {p:4d}   det = {np.linalg.det(V):12.4g}   slogdet -> {logdet:.4f}")

At $n=10, p\le 4$ none of this will actually overflow — but the marker is looking for code that
is *right*, and `slogdet` / `gammaln` cost nothing. Also prefer solving to inverting:
`np.linalg.solve(V, X.T @ y)` over `np.linalg.inv(V) @ X.T @ y`.

## Trap 2 — normalising three log-scale numbers

Part (ii) needs $\Prob(M_i\mid\y)$ from three log marginal likelihoods. The naive route
exponentiates first, which is exactly when you lose everything.

In [ ]:
log_ml = np.array([-812.4, -809.1, -810.7])   # invented, but a realistic magnitude

naive = np.exp(log_ml)
print("naive exp:      ", naive, "-> sum =", naive.sum())

stable = np.exp(log_ml - log_ml.max())
stable /= stable.sum()
print("shift then exp: ", stable, "-> sum =", stable.sum())

**Q1.** Why is subtracting the maximum legitimate — what makes the answer unchanged?

**Q2.** Your scaffold in `code/q1_model_choice.py` already does this. Make sure you can
justify the line rather than just keeping it.

## How to know your implementation is correct

Four checks, cheap to run, in increasing order of strength. The last one is the real test.

In [ ]:
# 1. Posterior model probabilities must sum to 1. (Necessary, very weak -- normalising
#    forces it even if every log marginal likelihood is wrong.)

# 2. Sensitivity: nudge lambda and re-run. The preferred model should not flip for a tiny
#    change. If it does, either the evidence is genuinely marginal (worth saying in part e)
#    or something is unstable.

# 3. Internal consistency: recompute m_n two ways and compare.
#       m_n = V_n X^T y      with V_n formed explicitly
#       m_n = solve(lambda I + X^T X,  X^T y)
#    These must agree to ~1e-12.

# 4. STRONGEST -- brute-force the integral for one model and compare.
#    For M1 (p = 2) the marginal likelihood is a 3-dimensional integral over
#    (beta_0, beta_1, sigma^2). Evaluate the integrand p(y | beta, sigma^2) p(beta, sigma^2)
#    on a grid, integrate numerically, take the log, and compare with your closed form.
#    Agreement to 2-3 decimals is convincing; disagreement localises the bug.

print("Implement check 4 below once your closed form exists.")

**Q3.** Check 4 is worth actually doing — it is the only one that can catch a wrong constant.
A dropped $2\pi$ or a factor of $\tfrac12$ shifts every log marginal likelihood by the *same*
amount. Ask yourself: would such a bug change your answer to part (ii)? To part (iii)? That
tells you how much the constants matter here — and it is a good sentence for the write-up.

---
## Your turn

Implement in `code/q1_model_choice.py` between the `# <<q1c` / `# >>q1c` markers — the report
pulls that block in automatically, so nothing needs copying.

Then export the numbers via `write_results` and cite them with `\result{...}` rather than
typing them into the prose.

Sketch of what to write for the prose around the code:

*(your text here — one or two sentences saying what the code does, then the three log marginal
likelihoods, the three posterior probabilities, and which model wins)*